# PyTorch LSTM 기반 항공 승객 수 예측 실습

이 노트북은 항공 승객 수 시계열 데이터를 사용하여 **과거 12개월 데이터로 다음 1개월 승객 수를 예측하는 LSTM 회귀 모델**을 구현합니다.

구성은 다음 순서로 진행됩니다.

1. 필요한 패키지 불러오기
2. 하이퍼파라미터 설정
3. 데이터 불러오기
4. 데이터 정규화
5. 시계열 입력/정답 데이터 생성
6. 학습 데이터와 테스트 데이터 분리
7. PyTorch Tensor 및 DataLoader 생성
8. LSTM 모델 정의
9. 손실 함수와 최적화 알고리즘 설정
10. 모델 학습
11. 모델 평가 및 예측값 복원
12. 예측 결과 시각화


## 1. 패키지 임포트

아래 코드는 데이터 처리, 시각화, 전처리, 딥러닝 모델 학습에 필요한 라이브러리를 불러옵니다.


In [ ]:
# 운영체제 관련 기능을 사용하기 위한 표준 라이브러리입니다.
# 예를 들어 파일이 존재하는지 확인할 때 사용합니다.
import os

# 학습 시간을 측정하기 위해 time 함수를 불러옵니다.
from time import time

# numpy는 배열 기반 수치 계산을 빠르게 처리하기 위해 사용하는 라이브러리입니다.
import numpy as np

# pandas는 CSV 파일을 읽고 표 형태의 데이터를 처리하기 위해 사용하는 라이브러리입니다.
import pandas as pd

# matplotlib.pyplot은 선 그래프, 산점도 등 기본 그래프를 그릴 때 사용하는 라이브러리입니다.
import matplotlib.pyplot as plt

# MinMaxScaler는 데이터를 0과 1 사이 범위로 변환하는 정규화 도구입니다.
from sklearn.preprocessing import MinMaxScaler

# mean_squared_error는 예측값과 실제값 사이의 평균제곱오차를 계산하는 평가 함수입니다.
from sklearn.metrics import mean_squared_error

# torch는 PyTorch의 핵심 라이브러리로 텐서 계산과 딥러닝 학습을 담당합니다.
import torch

# torch.nn은 신경망 계층, 손실 함수, 활성화 함수 등을 제공하는 모듈입니다.
import torch.nn as nn

# TensorDataset은 입력 데이터와 정답 데이터를 하나의 데이터셋으로 묶는 도구입니다.
# DataLoader는 데이터셋을 미니배치 단위로 나누어 반복적으로 공급하는 도구입니다.
from torch.utils.data import TensorDataset, DataLoader

# numpy 배열 출력 시 소수점 자릿수를 3자리로 제한하여 결과를 보기 쉽게 만듭니다.
np.set_printoptions(precision=3, suppress=True)

# numpy 난수 시드를 고정하여 매번 비슷한 실험 결과가 나오도록 합니다.
np.random.seed(42)

# PyTorch 난수 시드를 고정하여 모델 초기 가중치가 매번 비슷하게 생성되도록 합니다.
torch.manual_seed(42)

# GPU가 사용 가능하면 GPU(cuda)를 사용하고, 그렇지 않으면 CPU를 사용합니다.
# 구글 코랩에서 GPU 런타임을 켜면 cuda가 선택될 수 있습니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 사용 중인 학습 장치를 출력합니다.
print("사용 장치:", device)


## 2. 하이퍼파라미터 설정

하이퍼파라미터는 모델이 자동으로 학습하는 값이 아니라, 사람이 직접 정하는 학습 설정값입니다.


In [ ]:
# 과거 몇 개월의 데이터를 입력으로 사용할지 정합니다.
# 12로 설정하면 과거 12개월 승객 수를 보고 다음 1개월 승객 수를 예측합니다.
PAST_MONTHS = 12

# 전체 데이터 중 학습 데이터로 사용할 비율입니다.
# 0.8은 앞쪽 80% 데이터를 학습에 사용하고, 뒤쪽 20% 데이터를 테스트에 사용한다는 의미입니다.
TRAIN_RATIO = 0.8

# LSTM의 은닉 상태 크기입니다.
# 값이 클수록 모델이 더 복잡한 패턴을 표현할 수 있지만, 학습 시간이 늘고 과적합 위험도 증가합니다.
HIDDEN_SIZE = 300

# 입력 특성 수입니다.
# 여기서는 한 달의 승객 수 하나만 입력으로 사용하므로 1입니다.
INPUT_SIZE = 1

# 출력값 개수입니다.
# 다음 1개월 승객 수 하나를 예측하므로 1입니다.
OUTPUT_SIZE = 1

# 전체 학습 반복 횟수입니다.
# epoch는 전체 학습 데이터를 처음부터 끝까지 한 번 학습하는 단위입니다.
EPOCHS = 300

# 미니배치 크기입니다.
# 한 번의 가중치 업데이트에 사용할 샘플 개수를 의미합니다.
BATCH_SIZE = 64

# 학습률입니다.
# 모델 가중치를 한 번 수정할 때 얼마나 크게 수정할지 결정합니다.
LEARNING_RATE = 0.001

# 설정값을 확인하기 위해 출력합니다.
print("과거 입력 개월 수:", PAST_MONTHS)
print("학습 데이터 비율:", TRAIN_RATIO)
print("LSTM 은닉 상태 크기:", HIDDEN_SIZE)
print("학습 반복 횟수:", EPOCHS)
print("배치 크기:", BATCH_SIZE)
print("학습률:", LEARNING_RATE)


## 3. 데이터 불러오기

기본 실행 방식은 `airline.csv` 파일을 노트북과 같은 위치에 두고 실행하는 것입니다.

파일이 없을 경우에는 공개 CSV 주소에서 데이터를 읽어오도록 구성했습니다.


In [ ]:
# 사용할 CSV 파일명을 지정합니다.
# 직접 파일을 업로드한 경우에는 이 파일명이 현재 작업 폴더에 있어야 합니다.
CSV_PATH = "airline.csv"

# 로컬 파일이 없을 때 사용할 공개 데이터 주소입니다.
# 구글 코랩에서 실습할 때 파일 업로드 없이 바로 실행할 수 있도록 예비 경로로 사용합니다.
FALLBACK_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"

# 현재 작업 폴더에 airline.csv 파일이 존재하는지 확인합니다.
if os.path.exists(CSV_PATH):
    # 파일이 있으면 로컬 CSV 파일을 읽습니다.
    # usecols=[1]은 두 번째 열인 승객 수만 가져오겠다는 의미입니다.
    raw = pd.read_csv(CSV_PATH, header=None, usecols=[1])
else:
    # 파일이 없으면 공개 CSV 주소에서 데이터를 읽습니다.
    # 해당 데이터는 Month,Passengers 구조이므로 Passengers 열만 사용합니다.
    raw = pd.read_csv(FALLBACK_URL, usecols=["Passengers"])

# 컬럼명을 passengers로 통일합니다.
raw.columns = ["passengers"]

# 데이터가 숫자형인지 안전하게 변환합니다.
# errors="coerce"는 숫자로 변환할 수 없는 값이 있으면 NaN으로 바꿉니다.
raw["passengers"] = pd.to_numeric(raw["passengers"], errors="coerce")

# 결측치가 있는 행을 제거합니다.
# LSTM 학습에는 숫자 데이터가 필요하므로 비어 있는 값은 제거합니다.
raw = raw.dropna().reset_index(drop=True)

# 데이터 앞부분을 출력하여 정상적으로 읽혔는지 확인합니다.
print("데이터 샘플")
print(raw.head(12))

# 전체 데이터 크기를 출력합니다.
print("
전체 데이터 개수:", len(raw))

# 기초 통계량을 출력합니다.
print("
기초 통계량")
print(raw.describe())


## 4. 원본 데이터 시각화

시계열 데이터는 시간 순서에 따라 값이 어떻게 변하는지 먼저 확인하는 것이 중요합니다.


In [ ]:
# 그래프 크기를 설정합니다.
plt.figure(figsize=(10, 4))

# 월별 승객 수를 선 그래프로 그립니다.
plt.plot(raw["passengers"].values, label="Passengers")

# 그래프 제목을 설정합니다.
plt.title("Monthly Airline Passengers")

# x축 이름을 설정합니다.
plt.xlabel("Month Index")

# y축 이름을 설정합니다.
plt.ylabel("Passengers")

# 범례를 표시합니다.
plt.legend()

# 격자선을 표시하여 값의 변화를 더 쉽게 볼 수 있게 합니다.
plt.grid(True)

# 그래프를 출력합니다.
plt.show()


## 5. MinMax 정규화

신경망 모델은 입력값의 범위가 너무 크면 학습이 불안정해질 수 있습니다. 따라서 승객 수를 0과 1 사이 값으로 변환합니다.


In [ ]:
# MinMaxScaler 객체를 생성합니다.
# 이 도구는 데이터의 최솟값을 0, 최댓값을 1로 변환합니다.
scaler = MinMaxScaler(feature_range=(0, 1))

# raw 데이터프레임의 passengers 열을 2차원 배열 형태로 변환합니다.
# MinMaxScaler는 입력을 2차원 배열 형태로 받습니다.
passenger_values = raw[["passengers"]].values

# fit_transform은 최솟값과 최댓값을 학습한 뒤 실제 정규화 변환을 수행합니다.
scaled_values = scaler.fit_transform(passenger_values)

# 정규화 결과를 데이터프레임으로 변환하여 확인하기 쉽게 만듭니다.
scaled_df = pd.DataFrame(scaled_values, columns=["scaled_passengers"])

# 정규화된 데이터 앞부분을 출력합니다.
print("정규화 데이터 샘플")
print(scaled_df.head(12))

# 정규화 데이터의 최솟값과 최댓값을 출력합니다.
print("정규화 최솟값:", scaled_values.min())
print("정규화 최댓값:", scaled_values.max())


## 6. 시계열 입력 데이터와 정답 데이터 생성

LSTM에 넣을 입력 데이터는 `(샘플 수, 시간 길이, 특성 수)` 형태가 되어야 합니다.

예를 들어 과거 12개월을 입력으로 사용한다면 다음과 같이 구성됩니다.

- 입력 X: 1번째 달 ~ 12번째 달
- 정답 y: 13번째 달
- 다음 입력 X: 2번째 달 ~ 13번째 달
- 다음 정답 y: 14번째 달


In [ ]:
# 시계열 데이터를 입력 X와 정답 y로 변환하는 함수를 정의합니다.
def make_sequences(data, past_months):
    # 입력 데이터를 저장할 리스트입니다.
    X_list = []

    # 정답 데이터를 저장할 리스트입니다.
    y_list = []

    # 전체 데이터에서 과거 입력 길이만큼 제외한 위치까지 반복합니다.
    # 마지막 인덱스에서도 입력 past_months개와 다음 값 1개가 있어야 하기 때문입니다.
    for i in range(len(data) - past_months):
        # i 위치부터 i+past_months 전까지를 입력 데이터로 사용합니다.
        X_list.append(data[i : i + past_months])

        # i+past_months 위치의 값을 정답 데이터로 사용합니다.
        y_list.append(data[i + past_months])

    # 리스트를 numpy 배열로 변환합니다.
    X_array = np.array(X_list)

    # 리스트를 numpy 배열로 변환합니다.
    y_array = np.array(y_list)

    # 입력 배열과 정답 배열을 반환합니다.
    return X_array, y_array

# 정규화된 승객 수 데이터로 시계열 입력과 정답을 생성합니다.
X_data, y_data = make_sequences(scaled_values, PAST_MONTHS)

# 생성된 입력 데이터의 모양을 출력합니다.
print("X_data shape:", X_data.shape)

# 생성된 정답 데이터의 모양을 출력합니다.
print("y_data shape:", y_data.shape)

# 첫 번째 입력 데이터와 정답 데이터를 확인합니다.
print("첫 번째 입력 데이터")
print(X_data[0].flatten())
print("첫 번째 정답 데이터")
print(y_data[0])


## 7. 학습 데이터와 테스트 데이터 분리

시계열 데이터는 시간 순서가 중요합니다. 따라서 데이터를 무작위로 섞지 않고 앞부분은 학습, 뒷부분은 테스트에 사용합니다.


In [ ]:
# 전체 샘플 개수에 학습 비율을 곱하여 학습 데이터의 마지막 인덱스를 계산합니다.
train_size = int(len(X_data) * TRAIN_RATIO)

# 앞쪽 데이터를 학습용 입력 데이터로 사용합니다.
X_train = X_data[:train_size]

# 뒤쪽 데이터를 테스트용 입력 데이터로 사용합니다.
X_test = X_data[train_size:]

# 앞쪽 정답 데이터를 학습용 정답 데이터로 사용합니다.
y_train = y_data[:train_size]

# 뒤쪽 정답 데이터를 테스트용 정답 데이터로 사용합니다.
y_test = y_data[train_size:]

# 분리된 데이터의 모양을 출력합니다.
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


## 8. PyTorch Tensor와 DataLoader 생성

PyTorch 모델은 numpy 배열이 아니라 `torch.Tensor`를 입력으로 받습니다. 따라서 데이터를 Tensor로 변환해야 합니다.


In [ ]:
# 학습용 입력 데이터를 float32 타입의 PyTorch Tensor로 변환합니다.
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)

# 학습용 정답 데이터를 float32 타입의 PyTorch Tensor로 변환합니다.
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

# 테스트용 입력 데이터를 float32 타입의 PyTorch Tensor로 변환합니다.
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

# 테스트용 정답 데이터를 float32 타입의 PyTorch Tensor로 변환합니다.
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# 학습용 입력 Tensor와 정답 Tensor를 하나의 Dataset으로 묶습니다.
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# Dataset을 미니배치 단위로 공급하는 DataLoader를 생성합니다.
# 시계열 순서를 유지하기 위해 shuffle=False로 설정합니다.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Tensor 모양을 출력하여 모델 입력 형태가 올바른지 확인합니다.
print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)
print("X_test_tensor:", X_test_tensor.shape)
print("y_test_tensor:", y_test_tensor.shape)


## 9. PyTorch LSTM 모델 정의

아래 모델은 과거 12개월 승객 수 흐름을 LSTM으로 처리한 뒤, 마지막 시점의 은닉 출력을 사용하여 다음 달 승객 수를 예측합니다.


In [ ]:
# nn.Module을 상속받아 LSTM 회귀 모델 클래스를 정의합니다.
class AirlineLSTM(nn.Module):
    # __init__ 함수는 모델에 필요한 계층을 생성하는 초기화 함수입니다.
    def __init__(self, input_size, hidden_size, output_size):
        # 부모 클래스인 nn.Module의 초기화 기능을 실행합니다.
        super().__init__()

        # 입력 특성 수를 객체 변수로 저장합니다.
        self.input_size = input_size

        # LSTM 은닉 상태 크기를 객체 변수로 저장합니다.
        self.hidden_size = hidden_size

        # 출력값 개수를 객체 변수로 저장합니다.
        self.output_size = output_size

        # LSTM 계층을 생성합니다.
        # input_size는 한 시점에 들어오는 특성 수입니다.
        # hidden_size는 LSTM이 내부적으로 기억하는 정보의 크기입니다.
        # batch_first=True는 입력 형태를 (배치 크기, 시간 길이, 특성 수)로 사용한다는 뜻입니다.
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        # 완전연결층을 생성합니다.
        # LSTM의 마지막 시점 출력 hidden_size개를 최종 예측값 output_size개로 변환합니다.
        self.fc = nn.Linear(hidden_size, output_size)

        # 출력값을 0과 1 사이로 제한하기 위한 Sigmoid 함수입니다.
        # 정답 데이터가 MinMax 정규화되어 0과 1 사이에 있으므로 출력 범위도 맞춰줍니다.
        self.output_activation = nn.Sigmoid()

    # forward 함수는 입력 데이터가 모델 내부에서 어떤 순서로 계산되는지 정의합니다.
    def forward(self, x):
        # LSTM에 입력 데이터를 통과시킵니다.
        # lstm_out은 모든 시점의 출력이며, 모양은 (배치 크기, 시간 길이, 은닉 크기)입니다.
        # hidden과 cell 상태는 여기서는 직접 사용하지 않으므로 _로 받습니다.
        lstm_out, _ = self.lstm(x)

        # 모든 시점 중 마지막 시점의 출력만 가져옵니다.
        # 마지막 시점 출력은 과거 12개월 정보를 순차적으로 반영한 결과입니다.
        last_time_step = lstm_out[:, -1, :]

        # 마지막 시점 출력을 완전연결층에 넣어 예측값 1개로 변환합니다.
        output = self.fc(last_time_step)

        # 예측값을 0과 1 사이로 변환합니다.
        output = self.output_activation(output)

        # 최종 예측값을 반환합니다.
        return output

# 모델 객체를 생성합니다.
model = AirlineLSTM(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    output_size=OUTPUT_SIZE
)

# 모델을 CPU 또는 GPU 장치로 이동합니다.
model = model.to(device)

# 모델 구조를 출력합니다.
print(model)


## 10. 손실 함수와 최적화 알고리즘 설정

이 문제는 실제 승객 수에 가까운 숫자를 예측하는 회귀 문제입니다. 따라서 평균제곱오차 손실을 사용합니다.


In [ ]:
# 평균제곱오차 손실 함수를 생성합니다.
# 예측값과 실제값의 차이를 제곱한 뒤 평균을 내므로 회귀 문제에서 많이 사용됩니다.
criterion = nn.MSELoss()

# RMSprop 최적화 알고리즘을 생성합니다.
# model.parameters()는 학습 가능한 모든 가중치와 편향을 의미합니다.
optimizer = torch.optim.RMSprop(model.parameters(), lr=LEARNING_RATE)

# 손실 함수와 최적화 알고리즘을 출력합니다.
print("손실 함수:", criterion)
print("최적화 알고리즘:", optimizer)


## 11. 모델 학습

PyTorch에서는 학습 과정을 직접 반복문으로 작성합니다.

한 번의 미니배치 학습은 다음 순서로 진행됩니다.

1. 입력과 정답을 학습 장치로 이동
2. 모델 예측 수행
3. 손실값 계산
4. 기존 기울기 초기화
5. 역전파로 기울기 계산
6. 최적화 알고리즘으로 가중치 업데이트


In [ ]:
# epoch별 평균 학습 손실을 저장할 리스트입니다.
train_losses = []

# 학습 시작 시간을 기록합니다.
start_time = time()

# 학습 시작 메시지를 출력합니다.
print("학습 시작")

# 지정한 epoch 수만큼 전체 학습 데이터를 반복 학습합니다.
for epoch in range(1, EPOCHS + 1):
    # 모델을 학습 모드로 전환합니다.
    # 학습 모드는 가중치 업데이트가 이루어지는 상태입니다.
    model.train()

    # 한 epoch 동안의 손실 합계를 저장할 변수를 0으로 초기화합니다.
    epoch_loss_sum = 0.0

    # DataLoader에서 미니배치 단위로 입력과 정답을 가져옵니다.
    for batch_X, batch_y in train_loader:
        # 입력 미니배치를 현재 학습 장치로 이동합니다.
        batch_X = batch_X.to(device)

        # 정답 미니배치를 현재 학습 장치로 이동합니다.
        batch_y = batch_y.to(device)

        # 이전 반복에서 계산된 기울기를 초기화합니다.
        # PyTorch는 기울기를 자동으로 누적하므로 매 반복마다 초기화해야 합니다.
        optimizer.zero_grad()

        # 모델에 입력 데이터를 넣어 예측값을 계산합니다.
        predictions = model(batch_X)

        # 예측값과 실제 정답 사이의 손실값을 계산합니다.
        loss = criterion(predictions, batch_y)

        # 손실값을 기준으로 모델 가중치에 대한 기울기를 계산합니다.
        loss.backward()

        # 계산된 기울기를 사용하여 모델 가중치를 업데이트합니다.
        optimizer.step()

        # 현재 미니배치 손실에 미니배치 샘플 수를 곱해 누적합니다.
        epoch_loss_sum += loss.item() * batch_X.size(0)

    # 전체 학습 샘플 수로 나누어 epoch 평균 손실을 계산합니다.
    epoch_loss = epoch_loss_sum / len(train_dataset)

    # 계산된 평균 손실을 리스트에 저장합니다.
    train_losses.append(epoch_loss)

    # 첫 epoch와 50 epoch마다 학습 손실을 출력합니다.
    if epoch == 1 or epoch % 50 == 0:
        # 현재 학습 진행 상황을 출력합니다.
        print(f"Epoch [{epoch:3d}/{EPOCHS}] - Train MSE Loss: {epoch_loss:.6f}")

# 학습 종료 시간을 기록합니다.
end_time = time()

# 전체 학습 시간을 출력합니다.
print(f"학습 완료: {end_time - start_time:.2f}초")


## 12. 학습 손실 시각화

손실값이 점점 감소하면 모델이 학습 데이터의 패턴을 점차 잘 학습하고 있다는 뜻입니다.


In [ ]:
# 그래프 크기를 설정합니다.
plt.figure(figsize=(10, 4))

# epoch별 학습 손실을 선 그래프로 그립니다.
plt.plot(train_losses, label="Train MSE Loss")

# 그래프 제목을 설정합니다.
plt.title("Training Loss Curve")

# x축 이름을 설정합니다.
plt.xlabel("Epoch")

# y축 이름을 설정합니다.
plt.ylabel("MSE Loss")

# 범례를 표시합니다.
plt.legend()

# 격자선을 표시합니다.
plt.grid(True)

# 그래프를 출력합니다.
plt.show()


## 13. 모델 평가

평가 단계에서는 가중치를 업데이트하지 않습니다. 따라서 `torch.no_grad()`를 사용하여 기울기 계산을 중지합니다.


In [ ]:
# 모델을 평가 모드로 전환합니다.
model.eval()

# 테스트 입력 Tensor를 현재 장치로 이동합니다.
X_test_device = X_test_tensor.to(device)

# 테스트 정답 Tensor를 현재 장치로 이동합니다.
y_test_device = y_test_tensor.to(device)

# 평가에서는 기울기 계산이 필요 없으므로 no_grad 영역을 사용합니다.
with torch.no_grad():
    # 테스트 데이터에 대한 예측값을 계산합니다.
    test_predictions = model(X_test_device)

    # 테스트 데이터의 평균제곱오차 손실을 계산합니다.
    test_loss = criterion(test_predictions, y_test_device)

# 테스트 손실값을 출력합니다.
print(f"테스트 MSE Loss: {test_loss.item():.6f}")


## 14. 예측값을 원래 승객 수 단위로 복원

모델은 정규화된 값을 예측합니다. 사람이 해석하기 쉬운 승객 수 단위로 보기 위해 원래 스케일로 되돌립니다.


In [ ]:
# 예측 Tensor를 CPU로 이동한 뒤 numpy 배열로 변환합니다.
pred_scaled = test_predictions.detach().cpu().numpy()

# 테스트 정답 Tensor를 CPU의 numpy 배열로 변환합니다.
truth_scaled = y_test_tensor.detach().cpu().numpy()

# 정규화된 예측값을 원래 승객 수 단위로 복원합니다.
pred_original = scaler.inverse_transform(pred_scaled)

# 정규화된 실제값을 원래 승객 수 단위로 복원합니다.
truth_original = scaler.inverse_transform(truth_scaled)

# 그래프와 표 출력이 편하도록 1차원 배열로 변환합니다.
pred_original_1d = pred_original.flatten()
truth_original_1d = truth_original.flatten()

# 원래 단위 기준 평균제곱오차를 계산합니다.
original_mse = mean_squared_error(truth_original_1d, pred_original_1d)

# 원래 단위 기준 평균제곱근오차를 계산합니다.
original_rmse = np.sqrt(original_mse)

# 평가 지표를 출력합니다.
print(f"원래 승객 수 단위 MSE: {original_mse:.3f}")
print(f"원래 승객 수 단위 RMSE: {original_rmse:.3f}")

# 예측값과 실제값 일부를 표로 비교합니다.
result_df = pd.DataFrame({
    "truth": truth_original_1d,
    "prediction": pred_original_1d,
    "error": truth_original_1d - pred_original_1d
})

# 결과 앞부분을 출력합니다.
print(result_df.head(10))


## 15. 예측 결과 시각화

아래 그래프에서 실제값 선과 예측값 선이 가까울수록 모델이 테스트 구간의 흐름을 잘 예측한 것입니다.


In [ ]:
# 그래프 크기를 설정합니다.
plt.figure(figsize=(10, 5))

# 실제 승객 수를 선 그래프로 그립니다.
plt.plot(truth_original_1d, label="Truth")

# 모델 예측 승객 수를 선 그래프로 그립니다.
plt.plot(pred_original_1d, label="Prediction")

# 그래프 제목을 설정합니다.
plt.title("Airline Passenger Prediction")

# x축 이름을 설정합니다.
plt.xlabel("Test Month Index")

# y축 이름을 설정합니다.
plt.ylabel("Passengers")

# 범례를 표시합니다.
plt.legend()

# 격자선을 표시합니다.
plt.grid(True)

# 그래프를 출력합니다.
plt.show()


## 16. 다음 1개월 승객 수 예측

마지막 12개월 데이터를 입력으로 사용하면 데이터 이후의 다음 1개월 승객 수를 예측할 수 있습니다.


In [ ]:
# 전체 정규화 데이터 중 마지막 12개월 값을 가져옵니다.
last_sequence = scaled_values[-PAST_MONTHS:]

# 모델 입력 형태인 (배치 크기, 시간 길이, 특성 수)로 변환합니다.
last_sequence_tensor = torch.tensor(last_sequence.reshape(1, PAST_MONTHS, INPUT_SIZE), dtype=torch.float32)

# 입력 Tensor를 현재 장치로 이동합니다.
last_sequence_tensor = last_sequence_tensor.to(device)

# 모델을 평가 모드로 전환합니다.
model.eval()

# 다음 달 예측에서는 기울기 계산이 필요 없으므로 no_grad를 사용합니다.
with torch.no_grad():
    # 마지막 12개월 데이터를 사용하여 다음 1개월 값을 예측합니다.
    next_month_scaled = model(last_sequence_tensor)

# 예측 Tensor를 CPU의 numpy 배열로 변환합니다.
next_month_scaled_np = next_month_scaled.detach().cpu().numpy()

# 정규화된 예측값을 원래 승객 수 단위로 복원합니다.
next_month_prediction = scaler.inverse_transform(next_month_scaled_np)

# 다음 1개월 예측 승객 수를 출력합니다.
print(f"다음 1개월 예측 승객 수: {next_month_prediction[0, 0]:.2f}")


## 17. 전체 흐름 정리

이 노트북의 핵심 흐름은 다음과 같습니다.

1. 월별 항공 승객 수 데이터를 불러옵니다.
2. 데이터를 0과 1 사이로 정규화합니다.
3. 과거 12개월을 입력으로, 다음 1개월을 정답으로 구성합니다.
4. 시계열 순서를 유지한 채 학습 데이터와 테스트 데이터를 나눕니다.
5. 데이터를 PyTorch Tensor로 변환하고 DataLoader로 묶습니다.
6. LSTM 모델을 정의하고 평균제곱오차 손실로 학습합니다.
7. 테스트 데이터에서 예측값과 실제값을 비교합니다.
8. 마지막 12개월 데이터로 다음 1개월 승객 수를 예측합니다.
